> **Research record.** Cleaned copy of `900-features-stat-areas/100-create-dataset-monthly.ipynb` (data checks, dead ends and unlabelled plots removed; outputs cleared).
> Statistical area x month dataset: residents' remote-work level, POI accessibility, census attributes and distance to the main business district.
>
> Requires the authors' BigQuery project and the full GPS dataset, which are not public; the notebook is kept as a record of the analysis behind the paper. Run it with its own folder as the working directory and `settings.ini` at the repository root. For a runnable walk-through on synthetic data see `demo/`.

In [ ]:
import pandas as pd
from geofunctions import utils
import geopandas as gpd

## Description
This notebook aggregates residents' remote-work level, POI accessibility and census attributes by statistical area (2022 boundaries) and month.

In [ ]:
MONTHLY_DATASET_TABLE= "phd_datasets.sa_dataset_monthly_20250610"

In [ ]:
utils.read_query_to_dataframe(f"""
CREATE OR REPLACE TABLE {MONTHLY_DATASET_TABLE} as
SELECT 
sa.yishuv_stat_2022,sa.main_function_txt, b.*except(yishuv_stat_2022,main_function_txt),t.*except(yishuv_stat_2022),
sa.wrkoutloc_pcnt, sa.employeesannual_medwage, sa.empl_pcnt, sa.pop_approx, sa.employeeswage_decile9up, sa.computer_avg,
c.IndstJ_pcnt as itcom_prcnt,
IndstK_pcnt as finance_prcnt,
IndstM_pcnt as professional_prcnt,
hh_Dati_pcnt+hh_Haredi_pcnt as religous_prcnt,
st_area(sa.geometry) sa_area,
ST_DISTANCE(
    ST_CENTROID(sa.geometry),
    ST_GEOGPOINT(34.79188915373317, 32.07472355010219)
  ) AS distance_to_azrieli
FROM `phd-habidatum.phd_census.all_sa_main_features` sa
left join  phd_datasets.osm_pois_sa_agg t -- pois
using (yishuv_stat_2022)
left join phd_datasets.remote_work_agg_sa_20250321 b --rw
on cast(t.yishuv_stat_2022 as string)=b.yishuv_stat_2022
left join phd_census.census_2022 c
on LocalityCode	= sa.semel_yishuv
and StatArea=sa.stat_2022""")

### Add more remote work level features in the vicinity (3 neighbors+its own value)

In [ ]:
NEIGHBORS_TABLE= "phd-habidatum.phd_adm.sa_3knn_monthly"

WORKDAYS_THRESH=7

In [ ]:
utils.read_query_to_dataframe(f"""
CREATE OR REPLACE TABLE {MONTHLY_DATASET_TABLE} AS

WITH user_data_with_yishuv AS (
  -- Attach each user to their yishuv_stat_2022
  SELECT 
    b.identifier,
    b.month,
    t.yishuv_stat_2022,
    flag_remote_work_smoothed_max_thresh,
    remote_work_days_min_work,
    total_work_days,
    undefined_days,
    office_day_counts
  FROM `phd-habidatum.phd_users_locations.users_remote_work_monthly` b
  JOIN `phd-habidatum.phd_users_locations.work_locations` a
    ON b.identifier = a.identifier AND b.month = a.month
  JOIN `phd-habidatum.phd_adm.geohashes_with_sa_gushdan` t
    ON a.geohash_home = t.geohash
  WHERE total_work_days >= {WORKDAYS_THRESH}
),

joined_data AS (
  -- Join user data with neighbors
  SELECT 
    en.yishuv_stat_2022 AS focal_yishuv,
    u.month,
    flag_remote_work_smoothed_max_thresh,
    remote_work_days_min_work,
    total_work_days,
    undefined_days,
    office_day_counts,
    identifier
  FROM {NEIGHBORS_TABLE} en
  JOIN user_data_with_yishuv u
    ON en.knn_neighbor = u.yishuv_stat_2022
    and DATE(en.month)=u.month
),
tab as (

SELECT 
  month,
  focal_yishuv AS yishuv_stat_2022,
  AVG(flag_remote_work_smoothed_max_thresh) AS avg_remote_work_vicinity,
  AVG(SAFE_DIVIDE(remote_work_days_min_work, total_work_days - undefined_days)) AS share_remote_work_and_unident_days,
  AVG(SAFE_DIVIDE(remote_work_days_min_work, total_work_days)) AS share_remote_work_days,
  AVG(SAFE_DIVIDE(office_day_counts, total_work_days)) AS share_office_days,
  count(identifier) AS num_homes_vicinity
FROM joined_data
GROUP BY month, yishuv_stat_2022
)

select s.*,t.* except(month, yishuv_stat_2022) from {MONTHLY_DATASET_TABLE} s
left join tab t
on s.yishuv_stat_2022 =  t.yishuv_stat_2022
AND DATE(s.month) = DATE(t.month)
    """)


## Add neighborhood type

In [ ]:
utils.read_query_to_dataframe(f"""create or replace table {MONTHLY_DATASET_TABLE}
                              as 
                              select t.*, neighborhood_type, amenities_1km_area_wght
                               from {MONTHLY_DATASET_TABLE} t
                   join phd_datasets.sa_neigborhoods_types
using (yishuv_stat_2022)""")

## Relationships with census variables

WFH captures low employment. This has to be account in the model

### Add commuting distance: TODO

In [ ]:
gdf = utils.make_gdf(utils.read_query_to_dataframe("select yishuv_stat_2022, geometry from phd-habidatum.phd_adm.statistical_areas_gushdan"), geometry="geometry")